# RuinedFooocus + Face Swap (Colab)

A fork of [runew0lf/RuinedFooocus](https://github.com/runew0lf/RuinedFooocus) with a **real InsightFace face swap** and **PyraCanny** added, running **inside the notebook** — no `gradio.live`, no public URL.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Leave the *Runtime version* on **Latest**.

**Two cells:**

1. **Setup** — installs, starts the server, downloads every model (~15–25 min first time). Prints progress the whole way.
2. **Show the UI** — embeds the interface. Re-run it any time the frame goes blank; it restarts the server if needed.

**How to generate:** the normal RuinedFooocus UI → **Advanced** tab → put a pose photo on the left, a face photo on the right, press Generate.

> `inswapper_128.onnx` is InsightFace's *non-commercial research* model. Personal use is fine; commercial use needs a licence.


In [ ]:
#@title 1. Setup - install, download everything  { display-mode: "form" }
#@markdown Run this once. When it finishes, run the **Show the UI** cell below.
#@markdown
#@markdown Leave the runtime version on **Latest**.
FORCE_REINSTALL = False #@param {type:"boolean"}
SKIP_CHECKPOINT = False #@param {type:"boolean"}

import os, sys, subprocess, time, urllib.request, json

REPO    = "https://github.com/ahmrafi22/RuinedFooocus.git"
APP_DIR = "/content/RuinedFooocus"
LOG     = "/content/fooocus.log"
PORT    = 7865
URL     = f"http://127.0.0.1:{PORT}/"


def rule(title=""):
    print("\n" + "=" * 62)
    if title:
        print(title)
        print("=" * 62)


def tail(n=60):
    if os.path.exists(LOG):
        return "\n".join(open(LOG, encoding="utf-8", errors="ignore").read().splitlines()[-n:])
    return "(no log)"


def up():
    try:
        with urllib.request.urlopen(URL, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False


# ---------------------------------------------------------------- 1. machine
rule("1/5  machine")
print("Python:", sys.version.split()[0])
try:
    import torch
    print("torch :", torch.__version__)
except ImportError:
    print("torch : (not installed yet)")

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode == 0:
    print("GPU   :", smi.stdout.strip())
else:
    print("GPU   : NOT FOUND -> Runtime > Change runtime type > T4 GPU")

# ---------------------------------------------------------------- 2. code
rule("2/5  code")
if os.path.isdir(APP_DIR):
    print("updating existing checkout...")
    subprocess.run(["git", "-C", APP_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, APP_DIR], check=True)
os.chdir(APP_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

# ---------------------------------------------------------------- 3. install
# NOTE: launch.py is started WITHOUT --offline. In launch.py that flag skips
# the whole pip install step, which is how onnxruntime went missing before.
rule("3/5  installing dependencies (first run only, several minutes)")

if os.path.exists("freezetorch") and not FORCE_REINSTALL:
    print("torch already frozen, skipping the install step")
else:
    open("freezetorch", "w").close()
    env = dict(os.environ)
    env["TORCH_PLATFORM"] = "cu124"

    log = open(LOG, "w")
    proc = subprocess.Popen(
        [sys.executable, "launch.py", "--nobrowser",
         "--listen", "127.0.0.1", "--port", str(PORT)],
        stdout=log, stderr=subprocess.STDOUT, env=env, cwd=APP_DIR,
    )
    print("started PID", proc.pid)

    # ------------------------------------------------------------ 4. wait
    rule("4/5  waiting for the UI to answer")
    print("live log below. This returns when the server is up.\n")

    deadline = time.time() + 60 * 40
    pos = 0
    ready = False

    while time.time() < deadline:
        if os.path.exists(LOG):
            text = open(LOG, encoding="utf-8", errors="ignore").read()
            if len(text) > pos:
                for line in text[pos:].splitlines():
                    line = line.strip()
                    if not line or ("%" in line and ("|" in line or "it/s" in line or "B/s" in line)):
                        continue
                    print("   ", line[:150])
                pos = len(text)

        if proc.poll() is not None:
            print("\nFAILED - the app exited with code", proc.poll())
            print("-" * 62)
            print(tail(60))
            raise SystemExit("see the log above")

        if up():
            ready = True
            break

        time.sleep(5)

    if not ready:
        print("\nFAILED - did not come up within 40 minutes")
        print("-" * 62)
        print(tail(60))
        raise SystemExit("see the log above")

    print("\nserver is up")

# ---------------------------------------------------------------- 5. models
rule("5/5  models")

CKPT = os.path.join(APP_DIR, "models/checkpoints/sd_xl_base_1.0_0.9vae.safetensors")
CKPT_URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"
            "resolve/main/sd_xl_base_1.0_0.9vae.safetensors")

if SKIP_CHECKPOINT:
    print("skipping the checkpoint (SKIP_CHECKPOINT is on)")
elif os.path.exists(CKPT) and os.path.getsize(CKPT) > 6e9:
    print("checkpoint already present")
else:
    print("downloading SDXL base 1.0 (~6.9 GB)...")

    def _progress(blocks, block_size, total):
        if total > 0 and blocks % 300 == 0:
            print(f"   {blocks*block_size/1e9:5.1f} / {total/1e9:5.1f} GB", end="\r")

    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    urllib.request.urlretrieve(CKPT_URL, CKPT, _progress)
    print(f"\n   saved ({os.path.getsize(CKPT)/1e9:.1f} GB)")

print("\nface swap models (843 MB, only if not cached):")
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import modules.faceswap as faceswap
faceswap.get_swapper()
faceswap.get_analyzer(640)
print(faceswap.status())

rule("setup complete")
print("Now run the cell below to show the UI.")

In [ ]:
#@title 2. Show the UI  { display-mode: "form" }
#@markdown Run the setup cell first. Re-run this any time the frame goes blank.

import os, time, urllib.request

PORT = 7865
URL  = f"http://127.0.0.1:{PORT}/"

# If the runtime was restarted, the server is gone - bring it back.
def up():
    try:
        with urllib.request.urlopen(URL, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if not up():
    import subprocess, sys
    print("server is not running, starting it...")
    env = dict(os.environ)
    env["TORCH_PLATFORM"] = "cu124"
    log = open("/content/fooocus.log", "w")
    subprocess.Popen(
        [sys.executable, "launch.py", "--nobrowser",
         "--listen", "127.0.0.1", "--port", str(PORT)],
        stdout=log, stderr=subprocess.STDOUT, env=env, cwd="/content/RuinedFooocus",
    )
    for _ in range(240):
        if up():
            break
        time.sleep(5)
    if not up():
        raise SystemExit("the server did not start - run the setup cell and check its log")

from google.colab import output as colab_output
colab_output.serve_kernel_port_as_iframe(PORT, path="/", height=900, width="100%")

print()
print("The UI is the frame above - the normal RuinedFooocus interface.")
print("Face + pose references live in the **Advanced** tab.")